In [1]:
import os 
import cv2
import glob
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import scipy.stats as stats
import seaborn as sns
import pandas as pd
from sklearn.metrics import roc_curve
import re
import math
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report,precision_score, recall_score, f1_score
import joblib
import time
from ultralytics import YOLO

FlashAttention is not available on this device. Using scaled_dot_product_attention instead.


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
import os
import cv2
import numpy as np
from ultralytics import YOLO

# === 設定 ===
data = "maesyori_img"
input_file = "/home/data/maesyori_img/collage_1.jpg"

# 出力ディレクトリ
detect_output_folder = f"/home/test_hozon/"
os.makedirs(detect_output_folder, exist_ok=True)

# モデルの読み込み
detection_model = YOLO('/home/YOLO/hukusuu_train/datasets/train7/weights/best.pt')

# === 処理関数 ===
def visualize_detection(image_path):
    # 物体検出
    results = detection_model.predict(image_path)

    # 元画像とサイズ
    orig_img = results[0].orig_img
    draw_img = orig_img.copy()
    img_h, img_w, _ = orig_img.shape

    # 4×6のマスに分割
    rows, cols = 4, 6
    cell_h, cell_w = img_h // rows, img_w // cols

    # bbox の描画とマス目番号の表示
    if results[0].boxes is not None:
        for i, box in enumerate(results[0].boxes.xyxy):
            start_x, start_y, end_x, end_y = map(int, box)
            center_x, center_y = (start_x + end_x) // 2, (start_y + end_y) // 2

            # マスの行・列番号（1始まり）
            row_idx = min(center_y // cell_h, rows - 1)
            col_idx = min(center_x // cell_w, cols - 1)

            # BBOX描画
            cv2.rectangle(draw_img, (start_x, start_y), (end_x, end_y), (0, 255, 0), 20)
            cv2.putText(draw_img, "shiitake", (start_x, start_y - 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

            # マス目番号を表示（黄っぽく）
            cell_text = f"({row_idx+1},{col_idx+1})"
            text_x = max(start_x + 5, 0)
            text_y = max(start_y + 15, 0)
            cv2.putText(draw_img, cell_text, (text_x, text_y),
                        cv2.FONT_HERSHEY_SIMPLEX, 15, (255, 255, 0), 30)

    # 保存
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    output_path = os.path.join(detect_output_folder, f"{base_name}_detected.jpg")
    cv2.imwrite(output_path, draw_img)
    print(f"検出画像を保存しました: {output_path}")
    # plt.figure(figsize=(12, 8))
    # plt.imshow(cv2.cvtColor(draw_img, cv2.COLOR_BGR2RGB))
    # plt.axis('off')
    # # plt.title('物体検出結果')
    # plt.show()

# === 実行 ===
visualize_detection(input_file)



image 1/1 /home/data/maesyori_img/collage_1.jpg: 320x640 24 shiitake_bboxs, 13.3ms
Speed: 1.4ms preprocess, 13.3ms inference, 0.3ms postprocess per image at shape (1, 3, 320, 640)
検出画像を保存しました: /home/test_hozon/collage_1_detected.jpg
